In [3]:
import pandas as pd

books = pd.read_csv("../Data/final_books.csv")

In [6]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k=None, # Return all scores = True doesn't work anymore so top_k None gives all
                      device="mps")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528688922524452},
  {'label': 'neutral', 'score': 0.005764600355178118},
  {'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'sadness', 'score': 0.0020923931151628494},
  {'label': 'disgust', 'score': 0.0016119939973577857},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

#### As we know that our description are a bit large and assigning a single emotion to a book doesn't make sense, so we will break the description about fullstops in it and then get emotion prediction for each one of those and create a function which return the maximum score of each label, iterating for each split description line

In [17]:
import numpy as np

emotion_labels = ["joy", "surprise", "neutral", "anger", "sadness", "disgust", "fear"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}


def calc_max_emotion_scores(predictions):
    pre_emotion_scores = {label: [] for label in emotion_labels}
    for pred in predictions:
        score_by_label = {p["label"]: p["score"] for p in pred}
        for label in emotion_labels:
            pre_emotion_scores[label].append(score_by_label[label])
    return {label: np.max(scores) for label, scores in pre_emotion_scores.items()}


from tqdm import tqdm

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calc_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [05:18<00:00, 16.30it/s]


In [18]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [19]:
emotion_books_df = pd.merge(books, emotions_df, on="isbn13")

In [20]:
emotion_books_df.to_csv("emotion_books.csv", index=False)